Next steps:

Get the product listings from the EDA and then run this code on the merged file.
Replace /kaggle/input/product-items/50_sampled_data.csv with your file and then run. Shd take around half day.

Also check if the meta data file has the required paths so that the images can be retrieved. Some debugging needed there.

In [ ]:
# Library imports

# 📦 Core Libraries
import numpy as np  # Linear algebra
import pandas as pd  # Data processing, CSV I/O
import os
import torch  # PyTorch for deep learning
from PIL import Image  # For image processing
import time
import glob

# 🔄 Transformers for Model and Processor
from transformers import (
    AutoProcessor, AutoModel,
    BlipProcessor, BlipForConditionalGeneration,
    PaliGemmaProcessor, PaliGemmaForConditionalGeneration
)

# 🧹 Data Handling and Progress
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm  # Progress bar
import re  # For regular expressions

# For inference only
from torch.amp import autocast


In [ ]:


# Merge the product listings with image metadata efficiently
# Define paths based on Kaggle's directory structure
dataset_path = "/kaggle/input"
images_metadata_path = os.path.join(dataset_path, "images-small", "images", "metadata", "images.csv")
product_items_path = os.path.join(dataset_path, "product-listing-data-capped", "abo_products_capped_10percent.csv")
output_path = "/kaggle/working/merged_product_images.csv"

print("✓ Path to image metadata:", images_metadata_path)
print("✓ Path to product listings:", product_items_path)

# Verify files exist
if not os.path.exists(images_metadata_path):
    raise FileNotFoundError(f"Image metadata file not found at: {images_metadata_path}")
if not os.path.exists(product_items_path):
    raise FileNotFoundError(f"Product listings file not found at: {product_items_path}")

# Step 1: Load image metadata
print("\n📊 Loading image metadata...")
image_df = pd.read_csv(images_metadata_path)
print(f"   Loaded {len(image_df):,} image entries")
print(f"   Sample of image_df:")
print(image_df.head())

# Step 2: Load product listings
print("\n📦 Loading product listings...")
product_df = pd.read_csv(product_items_path)
print(f"   Loaded {len(product_df):,} product listings")
print(f"   Columns in product_df: {product_df.columns.tolist()}")

# Find the main image ID column in product listings
main_image_col = None
candidate_columns = ["main_image_id", "image_id", "main_image"]
for col in candidate_columns:
    if col in product_df.columns:
        main_image_col = col
        break

if main_image_col is None:
    print("\n⚠️ Warning: Could not find main image ID column automatically.")
    print("   Available columns:", product_df.columns.tolist())
    print("   Assuming 'main_image_id' as the column name. Adjust this if needed.")
    main_image_col = "main_image_id"

# Step 3: Perform the merge
print(f"\n🔄 Merging datasets on {main_image_col}...")
# Keep only necessary columns from product_df
product_subset = product_df[["item_id", main_image_col]].copy()
print(f"   Product dataset shape: {product_subset.shape}")

# Merge datasets
merged_df = pd.merge(
    product_subset,
    image_df,
    left_on=main_image_col,
    right_on="image_id",
    how="inner"
)

# Step 4: Prepare final dataset - only the three columns needed
result_df = merged_df[["item_id", main_image_col, "path"]].copy()

# Step 5: Generate statistics
print("\n📊 Merge Statistics:")
print(f"   • Total products in input: {len(product_df):,}")
print(f"   • Unique product IDs in input: {product_df['item_id'].nunique():,}")
print(f"   • Products with matching images: {result_df['item_id'].nunique():,}")
print(f"   • Merge success rate: {result_df['item_id'].nunique() / product_df['item_id'].nunique() * 100:.2f}%")
print(f"   • Total rows in merged dataset: {len(result_df):,}")

# Step 6: Save to CSV
print(f"\n💾 Saving merged dataset to: {output_path}")
result_df.to_csv(output_path, index=False)
print("✅ Save complete!")

# Step 7: Show sample of the merged dataset
print("\n📋 Sample of merged dataset:")
print(result_df.head())

In [ ]:


# Once all chunks are done, merge them
print("Merging all chunk files...")


all_captions = []
chunk_files = sorted(glob.glob("/kaggle/working/image_captions_chunk_*.csv"))

if chunk_files:
    # Track success of each file read for cleanup decisions
    successful_reads = {}
    
    for chunk_file in tqdm(chunk_files, desc="Reading chunks"):
        try:
            chunk_df = pd.read_csv(chunk_file)
            all_captions.append(chunk_df)
            successful_reads[chunk_file] = True
        except Exception as e:
            print(f"⚠️ Error reading {chunk_file}: {e}")
            successful_reads[chunk_file] = False
    
    if all_captions:
        print(f"Concatenating {len(all_captions)} chunks...")
        final_df = pd.concat(all_captions, ignore_index=True)
        
        # Remove duplicates if any
        final_df = final_df.drop_duplicates(subset=["Image_ID"])
        
        # Write the master file
        master_file = "/kaggle/working/image_captions_master.csv"
        print(f"Writing master file with {len(final_df)} captions...")
        final_df.to_csv(master_file, index=False)
        print(f"✅ All done! Generated captions for {len(final_df)} images.")
        
        # Verify master file was created successfully
        if os.path.exists(master_file):
            # Cleanup: Delete individual chunk files
            print("Cleaning up: Removing individual chunk files...")
            deleted_count = 0
            for chunk_file, read_success in successful_reads.items():
                if read_success:  # Only delete files that were successfully read
                    try:
                        os.remove(chunk_file)
                        deleted_count += 1
                    except Exception as e:
                        print(f"⚠️ Error deleting {chunk_file}: {e}")
            print(f"✅ Deleted {deleted_count}/{len(successful_reads)} chunk files.")
        else:
            print("⚠️ Master file creation failed. Keeping chunk files as backup.")
    else:
        print("No valid chunk files found to merge.")
else:
    print("No chunk files found to merge.")

# Clean up state file
if os.path.exists("/kaggle/working/chunk_state.txt"):
    os.remove("/kaggle/working/chunk_state.txt")
    print("✅ Removed chunk state tracker file.")

In [ ]:
# # Replace with your actual file path
# file_path = "/kaggle/working/image_captions_master.csv"

# if os.path.exists(file_path):
#     os.remove(file_path)
#     print(f"✅ Deleted: {file_path}")
# else:
#     print(f"⚠️ File not found: {file_path}")


In [ ]:


device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
# No need for scaler during inference

# # When loading the model
# model = BlipForConditionalGeneration.from_pretrained(
#     model_name,
#     torch_dtype=torch.float16 if device == "cuda" else torch.float32,
#     low_cpu_mem_usage=True
# ).to(device)

In [ ]:
# ✅ Configuration Section: Define parameters
N_IMAGES = 398 
# N_IMAGES = 2005  
# N_IMAGES = None  # Set to None to use the entire dataset
MODEL_TYPE = "BLIP"  # Options: "BLIP", "OFA", "PaliGemma"
USE_LARGE_MODEL = False  # Applies only to BLIP
# batch_size = 8  # Adjust based on available memory
# batch_size = 32

In [ ]:
print("hello")

In [ ]:
#  Dataset existence check

# ✅ Dataset Paths
dataset_path = "/kaggle/input"  # Root of input datasets
# images_small_path = os.path.join(dataset_path, "images-small")
# Update these paths to match your structure
image_folder = os.path.join(dataset_path, "images-small", "images", "small")  # Folder containing the small images
# metadata_file = os.path.join(dataset_path, "images-small", "images", "metadata", "images.csv")  # Path to CSV file
# output_path = "/kaggle/working/merged_product_images.csv"
metadata_file = "/kaggle/working/merged_product_images.csv"

if os.path.exists(image_folder):
    print(f"✅ Dataset folder exists: {image_folder}")
else:
    raise FileNotFoundError("❌ Dataset folder not found!")

if os.path.exists(metadata_file):
    print(f"✅ Metadata file exists: {metadata_file}")
else:
    raise FileNotFoundError("❌ Metadata file not found!")


In [ ]:
# ✅ Load metadata from CSV instead of scanning folders
print("📊 Loading image metadata from CSV...")
df_metadata = pd.read_csv(metadata_file)
print(f"✅ Loaded metadata for {len(df_metadata)} images")

# **Can start from here as output file (merged file) from previous code has been uploaded to input**

In [ ]:
import os

# List of files you want to delete from /kaggle/working/
files_to_delete = [
    "enriched_products_with_captions.csv",
    "image_captions_chunk_0.csv",
    "image_captions_chunk_1.csv",
    "image_captions_chunk_2.csv",
    "image_captions_chunk_3.csv",
    "image_captions_chunk_4.csv",
    "image_captions_master.csv",
    "products_with_captions.csv"
]

# Attempt deletion
for file_name in files_to_delete:
    file_path = f"/kaggle/working/{file_name}"
    try:
        os.remove(file_path)
        print(f"🗑️ Deleted: {file_name}")
    except FileNotFoundError:
        print(f"⚠️ File not found: {file_name}")
    except Exception as e:
        print(f"❌ Error deleting {file_name}: {e}")


In [ ]:
# ✅ Load metadata from CSV instead of scanning folders

metadata_preloaded_file = "/kaggle/input/merged-input-with-selected-images/merged_product_images (1).csv"

print("📊 Loading image metadata from CSV...")
df_metadata = pd.read_csv(metadata_preloaded_file)
print(f"✅ Loaded metadata for {len(df_metadata)} images")

In [ ]:
#  Dataset existence check

# ✅ Dataset Paths
dataset_path = "/kaggle/input"  # Root of input datasets
# images_small_path = os.path.join(dataset_path, "images-small")
# Update these paths to match your structure
image_folder = os.path.join(dataset_path, "images-small", "images", "small")  # Folder containing the small images
# metadata_file = os.path.join(dataset_path, "images-small", "images", "metadata", "images.csv")  # Path to CSV file
# output_path = "/kaggle/working/merged_product_images.csv"


if os.path.exists(image_folder):
    print(f"✅ Dataset folder exists: {image_folder}")
else:
    raise FileNotFoundError("❌ Dataset folder not found!")




In [ ]:
# ✅ Create full paths by combining base folder with CSV paths
df_metadata['full_path'] = df_metadata['path'].apply(lambda x: os.path.join(image_folder, x))

# ✅ Verify paths exist (you might want to check a subset for speed)
sample_check = df_metadata.sample(min(100, len(df_metadata)))
valid_paths = [os.path.exists(path) for path in sample_check['full_path']]
if all(valid_paths):
    print("✅ Sample path verification successful")
else:
    print(f"⚠️ Warning: {valid_paths.count(False)} out of {len(valid_paths)} sampled paths don't exist")
    # Display some examples of invalid paths
    invalid_examples = sample_check[~pd.Series(valid_paths)]['full_path'].tolist()[:5]
    if invalid_examples:
        print("Examples of invalid paths:")
        for path in invalid_examples:
            print(f"  - {path}")
        
        # Try to fix paths if they follow a common pattern
        print("Attempting path correction...")
        # Example: Check if we need to strip a prefix or add one
        if len(invalid_examples) > 0:
            example_path = invalid_examples[0]
            basename = os.path.basename(example_path)
            possible_path = os.path.join(image_folder, basename)
            if os.path.exists(possible_path):
                print(f"✅ Path correction hint: Use basename only")
                df_metadata['full_path'] = df_metadata['path'].apply(lambda x: os.path.join(image_folder, os.path.basename(x)))
            else:
                print("⚠️ Could not automatically fix paths")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"

In [ ]:
# ✅ Configuration Section: Define parameters
# N_IMAGES = 398 
# N_IMAGES = 2005
# N_IMAGES = 10000 
N_IMAGES = None  # Set to None to use the entire dataset
MODEL_TYPE = "BLIP"  # Options: "BLIP", "OFA", "PaliGemma"
USE_LARGE_MODEL = False  # Applies only to BLIP
# batch_size = 8  # Adjust based on available memory
# batch_size = 32

In [ ]:
# ✅ Create sample subset (for faster testing)
if N_IMAGES and N_IMAGES < len(df_metadata):
    df_sample = df_metadata.sample(N_IMAGES, random_state=42).reset_index(drop=True)
    print(f"🧪 Sample size: {len(df_sample)} images")
else:
    df_sample = df_metadata.reset_index(drop=True)
    print("📋 Using full dataset (no sampling)")

# ✅ Optional Preview
print("📑 Preview of dataset:")
print(df_sample[['image_id', 'path', 'full_path']].head())

In [ ]:
# Model loading

# ✅ Enable GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

# ✅ Cache model files (to avoid re-downloading)
os.environ["HF_HOME"] = "/kaggle/working/huggingface_cache"

# ✅ Define Model Type and Load Processor/Model
MODEL_TYPE = "BLIP"  # Options: "BLIP", "OFA", "PaliGemma"
USE_LARGE_MODEL = False  # Applies only to BLIP

# 🚫 Define unwanted words to filter out people-related descriptions
UNWANTED_WORDS = {}
# UNWANTED_WORDS = {
#     "person", "man", "woman", "hand", "holding", "standing", "pointing",
#     "sitting", "posing", "position", "body", "face", "finger", "arm", 
#     "whiteboard", "marker", "background", "photo", "image", "shown", "displayed"
# }

# ✅ Load the appropriate model and processor
if MODEL_TYPE == "BLIP":
    model_name = (
        "Salesforce/blip-image-captioning-large" if USE_LARGE_MODEL 
        else "Salesforce/blip-image-captioning-base"
    )
    processor = BlipProcessor.from_pretrained(model_name, local_files_only=False)
    # model = BlipForConditionalGeneration.from_pretrained(model_name, local_files_only=False).to(device)
    # Load with more optimized parameters
    model = BlipForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,  # Use half precision
        low_cpu_mem_usage=True
    ).to(device)
    
    # Optional: turn off gradient calculation since we're only doing inference
    for param in model.parameters():
        param.requires_grad = False
    model.eval()  # Set to evaluation mode

elif MODEL_TYPE == "OFA":
    model_name = "OFA-Sys/ofa-huge"
    processor = AutoProcessor.from_pretrained(model_name, local_files_only=False)
    model = AutoModel.from_pretrained(model_name, local_files_only=False).to(device)

elif MODEL_TYPE == "PaliGemma":
    model_name = "google/paligemma-3b-mix-224"
    processor = PaliGemmaProcessor.from_pretrained(model_name, local_files_only=False)
    model = PaliGemmaForConditionalGeneration.from_pretrained(model_name, local_files_only=False).to(device)

else:
    raise ValueError("❌ Invalid MODEL_TYPE. Choose from: 'BLIP', 'OFA', or 'PaliGemma'.")

print(f"✅ Using model: {model_name} on device: {device}")


In [ ]:
def find_optimal_batch_size(model, processor, device, start_size=32, max_size=64, step=8):
    """Find the largest batch size that works with the current GPU memory"""
    print("Finding optimal batch size...")
    
    # Create a dummy dataset
    dummy_img = Image.new('RGB', (224, 224), color='white')
    current_size = start_size
    
    while current_size <= max_size:
        try:
            print(f"Trying batch size: {current_size}")
            # Create a batch of the current size
            dummy_batch = [dummy_img] * current_size
            
            # Process the batch
            with torch.no_grad():
                inputs = processor(images=dummy_batch, return_tensors="pt", padding=True).to(device)
                outputs = model.generate(**inputs, max_length=10, num_beams=2)
                
            # If we got here, the batch size works
            current_size += step
            torch.cuda.empty_cache()  # Clear cache between tests
        except RuntimeError as e:
            if "CUDA out of memory" in str(e):
                # We've found the limit, back off by one step
                optimal_size = max(current_size - step, 1)
                print(f"✅ Optimal batch size: {optimal_size}")
                torch.cuda.empty_cache()
                return optimal_size
            else:
                # Some other error occurred
                raise e
    
    # If we got here, even the max_size works
    print(f"✅ Optimal batch size: {current_size - step} (or higher)")
    return current_size - step

# Use dynamic batch size after loading the model
batch_size = find_optimal_batch_size(model, processor, device, start_size=16, max_size=64)

In [ ]:
# # Query-Aware Prompts

# # 🧠 Choose Prompt Mode
# PROMPT_MODE = "default"  # Options: default, search_query, search_keywords, short_name, marketing
# # PROMPT_MODE = "search_query"
# # PROMPT_MODE = "search_keywords"
# # PROMPT_MODE = "short_name"

# PROMPT_TEXTS = {
#     "default": "Describe only the main product in this image.",
#     # "search_query": "What is this product called when someone searches for it online?",
#     # "search_keywords": "List keywords a user might type when searching for this product.",
#     # "short_name": "Provide only the product name, as if for a product catalog entry.",
# }

# product_prompt = PROMPT_TEXTS[PROMPT_MODE]
# print(f"📣 Using prompt mode: {PROMPT_MODE} -> {product_prompt}")



In [ ]:
# # Create a PyTorch Dataset for Efficient Batch Processing

# Optimize the Dataset class for faster loading
class ImageMetadataDataset(Dataset):
    def __init__(self, metadata_df, transform=None):
        self.metadata_df = metadata_df
        self.transform = transform
        # Pre-cache file existence to avoid checking during training
        self._validate_paths()
        
    def _validate_paths(self):
        """Pre-validate paths to avoid issues during training"""
        print("Validating image paths...")
        valid_mask = []
        for idx, row in tqdm(self.metadata_df.iterrows(), total=len(self.metadata_df)):
            valid_mask.append(os.path.exists(row['full_path']))
        
        # Filter DataFrame to only include valid paths
        valid_count = sum(valid_mask)
        print(f"Found {valid_count}/{len(valid_mask)} valid images ({valid_count/len(valid_mask)*100:.1f}%)")
        self.metadata_df = self.metadata_df[valid_mask].reset_index(drop=True)

    def __len__(self):
        return len(self.metadata_df)

    def __getitem__(self, idx):
        row = self.metadata_df.iloc[idx]
        img_path = row['full_path']
        img_id = row['image_id']
        
        try:
            # Use PIL's convert to ensure consistent RGB format
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            return img_id, img_path, image
        except Exception as e:
            print(f"❌ Error loading image {img_path} (ID: {img_id}): {e}")
            # Return a placeholder black image instead of None to maintain batch integrity
            return img_id, img_path, Image.new('RGB', (224, 224), color='black')


In [ ]:
def collate_fn(batch):
    img_ids, img_paths, images = zip(*batch)  # Separate IDs, paths and images
    
    # Filter out None images but keep track of their positions
    valid_items = [(i, img_id, path, img) for i, (img_id, path, img) in enumerate(zip(img_ids, img_paths, images)) if img is not None]
    
    if len(valid_items) == 0:
        return None, None, None  # Return empty batch if all images failed
    
    indices, valid_ids, valid_paths, valid_images = zip(*valid_items)
    
    # Process images as a batch - keep tensors on CPU first
    inputs = processor(images=valid_images, return_tensors="pt", padding=True)
    
    # Move to device after processing (instead of in the processor call)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    return valid_ids, valid_paths, inputs

In [ ]:

# ✅ Create DataLoader with the metadata-aware dataset
dataset = ImageMetadataDataset(df_sample)  # Use the sample dataset

# dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
# Replace your current DataLoader with this optimized version
dataloader = DataLoader(
    dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    collate_fn=collate_fn,
    # num_workers=4,  # Parallel data loading
    num_workers=0,  #  Since you're running on Kaggle with a P100 GPU, you can simply set num_workers=0 in your DataLoader to avoid multiprocessing altogether. This often works well enough
    pin_memory=False  # To avoid the pinning error
)

print("✅ DataLoader ready.")

In [ ]:
    #     # Define different product prompts for evaluation
#     # Comment/uncomment the desired prompt before running the model
    
#     # product_prompt = "Describe only the main product in this image."
#     # product_prompt = "Ignore people and focus on the item shown."
#     # product_prompt = "Generate a product listing-style caption."
#     # product_prompt = "Describe the product shown in the image with its key features."
#     # product_prompt = "Describe the product in technical terms and specifications."
#     # product_prompt = "Provide a short product name only."
    
#     # product_prompt = "Describe only the main product in this image."  # Default prompt
#     product_prompt = "How would someone describe the main product in details if they searched for it online? Do not describe the background or setting of the main product."
#     # product_prompt = "Give only the product name or short description. Do not mention the background."
#     # product_prompt = "Write how someone might search for this product online, ignoring background or setting."

#     # Use the global product_prompt defined earlier
#     # No need to re-assign here


In [ ]:
# Generate Captions Using Batches

# ✅ Caption Cleaning Functions
def clean_caption(caption):
    caption = re.sub(r'\b(\w+\s+){1,3}(\1){2,}', r'\1', caption)
    words = caption.split()
    cleaned = []
    for i, word in enumerate(words):
        if i < 2 or word != words[i-1] or word != words[i-2]:
            cleaned.append(word)
    caption = ' '.join(cleaned)
    caption = re.sub(r'(and\s+a\s*)+$', '', caption)
    caption = re.sub(r'(connected\s+to\s*)+$', '', caption)
    if not caption.strip() or len(caption.split()) < 3:
        caption = "[undetermined product]"
    return caption.strip()

In [ ]:
def generate_captions(dataloader, save_every=500, csv_path="/kaggle/working/image_captions.csv", 
                      initial_batch_size=None):
    captions = []
    total_processed = 0
    batch_processed = 0
    current_batch_size = initial_batch_size or dataloader.batch_size
    
    # Resume logic
    if os.path.exists(csv_path):
        df_existing = pd.read_csv(csv_path)
        processed_ids = set(df_existing["Image_ID"].tolist())
        print(f"📂 Resuming from previous run. {len(processed_ids)} captions already saved.")
    else:
        processed_ids = set()
        
    # Base path to remove from the full path
    base_path = os.path.join(dataset_path, "images-small", "images", "small")
    
    # Track time for performance metrics
    start_time = time.time()
    last_time = start_time
    
    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Generating captions")):
        img_ids, img_paths, inputs = batch
        if inputs is None:
            continue
        
        try:
            with torch.no_grad():
                # Use autocast for mixed precision with correct API
                if use_amp:
                    with autocast(device_type='cuda'):
                        outputs = model.generate(**inputs, max_length=30, num_beams=5)
                else:
                    outputs = model.generate(**inputs, max_length=30, num_beams=5)
                    
            decoded_captions = [processor.decode(out, skip_special_tokens=True) for out in outputs]
            
            for img_id, img_path, caption in zip(img_ids, img_paths, decoded_captions):
                if img_id in processed_ids:
                    continue
                    
                # Extract relative path
                relative_path = img_path.replace(base_path, "")
                if not relative_path.startswith('/'):
                    relative_path = '/' + relative_path
                    
                # Filter and clean caption
                filtered_caption = " ".join([word for word in caption.split() 
                                             if word.lower() not in UNWANTED_WORDS])
                cleaned_caption = clean_caption(filtered_caption)
                
                captions.append((img_id, relative_path, cleaned_caption))
                total_processed += 1
                batch_processed += 1
                
            # Periodically save progress
            if batch_processed >= save_every:
                df_chunk = pd.DataFrame(captions, columns=["Image_ID", "Image_Path", "Caption"])
                df_chunk.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)
                
                # Calculate and display performance metrics
                current_time = time.time()
                elapsed = current_time - last_time
                images_per_sec = batch_processed / elapsed
                
                print(f"💾 Saved batch {batch_idx+1}/{len(dataloader)}, "
                      f"Total: {total_processed} images, "
                      f"Speed: {images_per_sec:.2f} img/sec")
                      
                captions = []
                batch_processed = 0
                last_time = current_time
                
            # Periodically clear CUDA cache to prevent OOM errors
            if use_amp and (batch_idx + 1) % 20 == 0:
                torch.cuda.empty_cache()
                
        except RuntimeError as e:
            if "CUDA out of memory" in str(e):
                # OOM error - reduce batch size and retry
                if current_batch_size <= 1:
                    print("❌ Cannot reduce batch size further (already at 1). Saving progress and exiting.")
                    break
                
                # Save current progress
                if captions:
                    df_chunk = pd.DataFrame(captions, columns=["Image_ID", "Image_Path", "Caption"])
                    df_chunk.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)
                    print(f"💾 Saved {len(captions)} captions before batch size adjustment")
                    captions = []
                    batch_processed = 0
                
                # Reduce batch size by 25% and recreate dataloader
                torch.cuda.empty_cache()
                current_batch_size = max(1, int(current_batch_size * 0.75))
                print(f"⚠️ CUDA OOM detected. Reducing batch size to {current_batch_size}")
                
                # Return reduced batch size so caller can recreate dataloader
                return total_processed, current_batch_size, "OOM"
            else:
                # Other error - log and continue
                print(f"❌ Error processing batch {batch_idx}: {e}")
                continue
                
    # Save any remaining captions
    if captions:
        df_chunk = pd.DataFrame(captions, columns=["Image_ID", "Image_Path", "Caption"])
        df_chunk.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)
        
    # Final performance report
    total_time = time.time() - start_time
    print(f"✅ Caption generation complete. Total: {total_processed} captions in {total_time:.2f} sec "
          f"({total_processed/total_time:.2f} img/sec average)")
    
    return total_processed, current_batch_size, "COMPLETE"

In [ ]:
# Execute with better monitoring and error handling
try:
   # Set up interruption handling
    import signal
    import glob
    
    # Flag to track if interrupt has already been handled
    interrupt_handled = False
    
    def handle_interrupt(sig, frame):
        global interrupt_handled
        if interrupt_handled:
            return  # Skip if already handled
            
        interrupt_handled = True
        print("\n⚠️ Interrupted! Saving progress before exiting...")
        
        if 'captions' in locals() and len(captions) > 0:
            try:
                df_chunk = pd.DataFrame(captions, columns=["Image_ID", "Image_Path", "Caption"])
                df_chunk.to_csv(current_chunk_path, mode='a', header=not os.path.exists(current_chunk_path), index=False)
                print(f"💾 Saved {len(captions)} captions on exit")
                
                # Save chunk state
                with open("/kaggle/working/chunk_state.txt", "w") as f:
                    f.write(f"{current_chunk_idx}")
            except Exception as e:
                print(f"Error during cleanup: {e}")
        
        # Use os._exit instead of exit() to force immediate termination
        import os
        os._exit(0)
    
    # Register the interrupt handler
    signal.signal(signal.SIGINT, handle_interrupt)
    signal.signal(signal.SIGTERM, handle_interrupt)
    
    # Log GPU information before starting
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Define chunk size
    CHUNK_SIZE = 5000

    # OPTION 1: Process only N_IMAGES (sample dataset)
    # Comment this block if you want to process the full dataset
    dataset_to_process = df_sample  # Use the sample dataset created earlier with N_IMAGES
    print(f"Processing a SAMPLE of {len(dataset_to_process)} images")
    
    # OPTION 2: Process the full dataset
    # Uncomment this line to process the entire dataset instead of the sample
    # dataset_to_process = df_metadata
    # print(f"Processing the FULL dataset of {len(dataset_to_process)} images")

    # Calculate total chunks needed
    total_chunks = (len(dataset_to_process) + CHUNK_SIZE - 1) // CHUNK_SIZE
    
    # Determine where to resume from
    start_chunk = 0
    current_batch_size = batch_size  # Start with the optimal batch size we found
    
    # Check if we have a saved state file
    if os.path.exists("/kaggle/working/chunk_state.txt"):
        with open("/kaggle/working/chunk_state.txt", "r") as f:
            start_chunk = int(f.read().strip())
        print(f"📋 Resuming from chunk {start_chunk}")
    else:
        # Get list of already processed files to determine where to resume
        existing_chunks = glob.glob("/kaggle/working/image_captions_chunk_*.csv")
        processed_chunks = [int(f.split("_")[-1].split(".")[0]) for f in existing_chunks 
                           if f.split("_")[-1].split(".")[0].isdigit()]
        if processed_chunks:
            start_chunk = max(processed_chunks) + 1
            
    print(f"Processing dataset in {total_chunks} chunks, starting from chunk {start_chunk}")
    
    # Process in chunks
    for chunk_idx in range(start_chunk, total_chunks):
        current_chunk_idx = chunk_idx  # For the interrupt handler
        chunk_start = chunk_idx * CHUNK_SIZE
        chunk_end = min(chunk_start + CHUNK_SIZE, len(dataset_to_process))
        
        print(f"Processing chunk {chunk_idx + 1}/{total_chunks}: images {chunk_start} to {chunk_end}")
        
        # Create chunk dataset and dataloader
        df_chunk = dataset_to_process.iloc[chunk_start:chunk_end].reset_index(drop=True)
        
        # Keep trying with reducing batch size if OOM errors occur
        while True:
            chunk_dataset = ImageMetadataDataset(df_chunk)
            chunk_dataloader = DataLoader(
                chunk_dataset, 
                batch_size=current_batch_size, 
                shuffle=False, 
                collate_fn=collate_fn, 
                num_workers=0, 
                pin_memory=False
            )
            
            # Set current chunk path for the interrupt handler
            current_chunk_path = f"/kaggle/working/image_captions_chunk_{chunk_idx}.csv"
            
            # Generate captions for this chunk
            total_processed, new_batch_size, status = generate_captions(
                chunk_dataloader, 
                csv_path=current_chunk_path,
                initial_batch_size=current_batch_size
            )
            
            # Clear memory
            del chunk_dataset, chunk_dataloader
            torch.cuda.empty_cache()
            
            if status == "OOM":
                current_batch_size = new_batch_size
                print(f"Retrying with new batch size: {current_batch_size}")
            else:
                break
        
        # Save chunk state after successful completion
        with open("/kaggle/working/chunk_state.txt", "w") as f:
            f.write(f"{chunk_idx + 1}")
            
        print(f"✅ Completed chunk {chunk_idx + 1}/{total_chunks}")
    
    # Once all chunks are done, merge them
    print("Merging all chunk files...")
    all_captions = []
    chunk_files = sorted(glob.glob("/kaggle/working/image_captions_chunk_*.csv"))
    
    if chunk_files:
        for chunk_file in tqdm(chunk_files, desc="Reading chunks"):
            try:
                chunk_df = pd.read_csv(chunk_file)
                all_captions.append(chunk_df)
            except Exception as e:
                print(f"⚠️ Error reading {chunk_file}: {e}")
        
        if all_captions:
            print(f"Concatenating {len(all_captions)} chunks...")
            final_df = pd.concat(all_captions, ignore_index=True)
            
            # Remove duplicates if any
            final_df = final_df.drop_duplicates(subset=["Image_ID"])
            
            print(f"Writing master file with {len(final_df)} captions...")
            final_df.to_csv("/kaggle/working/image_captions_master.csv", index=False)
            print(f"✅ All done! Generated captions for {len(final_df)} images.")
        else:
            print("No valid chunk files found to merge.")
    else:
        print("No chunk files found to merge.")
    
    # Clean up state file
    if os.path.exists("/kaggle/working/chunk_state.txt"):
        os.remove("/kaggle/working/chunk_state.txt")
    
except Exception as e:
    print(f"❌ Error during execution: {e}")
    import traceback
    traceback.print_exc()

so images make up batches which make up chunks which make up the dataset to be processed?

Yes, that's exactly right! Your understanding of the processing hierarchy is correct:

Images: These are the individual product images from the Amazon Berkeley Objects dataset that you want to caption.
Batches: Groups of images (like 64 or 32) that are processed simultaneously through the neural network. Batching improves efficiency by leveraging GPU parallelism.
Chunks: Larger collections of images (like 80 or 1000) that are processed together before saving results. Chunks provide a balance between efficiency and recoverability.
Dataset: The entire collection of images you want to process, which could be either a sample (like your 398 images) or the full dataset (all 398,212 images).
The processing flow works like this:

The dataset is split into chunks
Each chunk is processed in batches
Results for each chunk are saved to a separate CSV file
After all chunks are processed, the individual CSV files are merged
This hierarchical approach gives you:

Efficiency through batch processing
Recoverability through chunk-based saving
Flexibility to process varying dataset sizes
Memory management by processing smaller portions at a time
Your setup with chunks of 80 images and dynamically determined batch sizes is a good approach for developing and testing the ShopTalk system, especially given the memory and time constraints.

In [ ]:
# Checking post electricity interruption

# import glob
# import os
# import pandas as pd

# # List all chunk files
# chunk_files = sorted(glob.glob("/kaggle/working/image_captions_chunk_*.csv"))
# print(f"Found {len(chunk_files)} chunk files:")
# for file in chunk_files:
#     file_size = os.path.getsize(file) / 1024  # Size in KB
#     print(f"  • {os.path.basename(file)}: {file_size:.2f} KB")
    
#     # Optional: Check row counts in each file
#     try:
#         df = pd.read_csv(file)
#         print(f"    - Contains {len(df)} rows")
#     except Exception as e:
#         print(f"    - Could not read file: {e}")

# # Check chunk state
# if os.path.exists("/kaggle/working/chunk_state.txt"):
#     with open("/kaggle/working/chunk_state.txt", "r") as f:
#         current_chunk = f.read().strip()
#     print(f"\nCurrent chunk being processed (from state file): {current_chunk}")
# else:
#     print("\nNo chunk state file found")

Need to do only for the n roughly 1.5 lac products selected! not the entire 398k

TODO: merge with the final product list item_id and main_image_id and then run this code.

Performance Analysis

Average processing speed: Your data shows consistent performance of ~2.18 images per second
Time per chunk of 80 images: ~36.5 seconds
Total processed images: 398 images in ~182 seconds (sum of all chunks)

Calculation for 400,000 Images
At the current rate of 2.18 images per second:

400,000 images ÷ 2.18 images/second = 183,486 seconds
183,486 seconds = 3,058 minutes = 50.97 hours = ~51 hours

That's approximately 2.1 days of continuous processing.
Optimization Considerations
You could reduce this time by:

Increasing batch size: Your current batch size might not be maximizing GPU utilization
Using larger chunks: Reduce overhead from starting/stopping chunks
Parallelization: Processing multiple chunks simultaneously if possible
Model optimization: Using a smaller or quantized model
Higher-end GPU: A more powerful GPU would process images faster

Practical Execution Plan
For a production run of this scale:

Start with a small sample (~1000 images) to verify everything works
Run the full dataset with chunks of 5,000-10,000 images
Set up monitoring to ensure the process continues without issues
Consider dividing the dataset across multiple machines if time is critical

Given the 2-day processing time, I'd recommend running this on a persistent cloud instance rather than a Kaggle notebook, which might time out during extended runs.RetryClaude can make mistakes. Please double-check responses.

In [ ]:
#  Final merge

# Import necessary libraries
import pandas as pd
import os

# Define input and output paths
# captions_path = "/kaggle/input/image-captioning-output/image_captions_master (5).csv"
captions_path = "/kaggle/input/final-image-captions/image_captions_master.csv"
products_path = "/kaggle/input/product-listing-data-capped/abo_products_capped_10percent.csv"
output_path = "/kaggle/working/enriched_products_with_captions.csv"

# Load the datasets
print("Loading datasets...")
captions_df = pd.read_csv(captions_path)
products_df = pd.read_csv(products_path)

# Print basic info about both datasets
print(f"Captions dataset: {len(captions_df)} rows")
print(f"Products dataset: {len(products_df)} rows")

# Display sample data to understand the structure
print("\nSample of captions dataset:")
print(captions_df.head(2))

print("\nSample of products dataset:")
print(products_df.head(2))

# Check column names to understand how to merge
print("\nCaption columns:", captions_df.columns.tolist())
print("Product columns:", products_df.columns.tolist())

# Identify the common key for merging
# Assuming the key is 'main_image_id' in products and 'Image_ID' in captions
# Adjust these column names if they're different in your datasets
print("\nPreparing to merge datasets...")

# First, rename the caption columns to match a better naming convention if needed
captions_df = captions_df.rename(columns={
    'Image_ID': 'image_id',
    'Caption': 'image_caption',
    'Image_Path': 'image_path'
})

# Now perform the merge
merged_df = pd.merge(
    products_df,
    captions_df[['image_id', 'image_caption', 'image_path']],
    on='image_id',
    how='left'
)

# Check merge result
print(f"\nMerged dataset: {len(merged_df)} rows")
print(f"Products with captions: {merged_df['image_caption'].notna().sum()}")
print(f"Products without captions: {merged_df['image_caption'].isna().sum()}")

# Save the merged dataset
merged_df.to_csv(output_path, index=False)
print(f"\nSaved enriched products dataset to: {output_path}")

# Additional analysis 
caption_coverage = (merged_df['image_caption'].notna().sum() / len(merged_df)) * 100
print(f"Caption coverage: {caption_coverage:.2f}%")

# If you want to save a clean version with only products that have captions
products_with_captions = merged_df.dropna(subset=['image_caption'])
clean_output_path = "/kaggle/working/products_with_captions.csv"
products_with_captions.to_csv(clean_output_path, index=False)
print(f"Saved {len(products_with_captions)} products with captions to: {clean_output_path}")



**# DATASET REVIEW**


In [ ]:
# # Check how many times each image_id appears in the products dataset
# image_id_counts = products_df['image_id'].value_counts()
# duplicated_image_ids = image_id_counts[image_id_counts > 1]

# print(f"Number of image IDs used by multiple products: {len(duplicated_image_ids)}")
# print(f"Total products using shared images: {duplicated_image_ids.sum()}")
# print("\nMost frequently reused images:")
# print(duplicated_image_ids.head(10))

In [ ]:
# # Find 5 image IDs that are used by multiple products
# multi_product_image_ids = duplicated_image_ids.head(5).index.tolist()

# # For each of these image IDs, show the corresponding products
# print("Examples of shared images across multiple products:")
# print("-" * 80)

# for i, image_id in enumerate(multi_product_image_ids, 1):
#     # Get all products that use this image ID
#     products_with_image = products_df[products_df['image_id'] == image_id]
    
#     # Display image ID and how many products use it
#     print(f"\nExample {i}: Image ID '{image_id}' is used by {len(products_with_image)} products")
    
#     # Show key information about these products
#     examples = products_with_image[['item_id', 'item_name']].head(3)
#     print(examples)
    
#     # If there are more than 3 products, indicate that
#     if len(products_with_image) > 3:
#         print(f"... and {len(products_with_image) - 3} more products")
    
#     print("-" * 80)

In [ ]:
# # Find 5 image IDs that are used by multiple products
# # Step 1: Count duplicates in merged_df (not products_df)
# duplicated_image_ids = merged_df['image_id'].value_counts()
# duplicated_image_ids = duplicated_image_ids[duplicated_image_ids > 1]

# multi_product_image_ids = duplicated_image_ids.head(5).index.tolist()

# print("Examples of shared images across multiple products (with image paths):")
# print("-" * 100)

# for i, image_id in enumerate(multi_product_image_ids, 1):
#     # Get all products that use this image ID
#     products_with_image = merged_df[merged_df['image_id'] == image_id]
    
#     # Display image ID and how many products use it
#     print(f"\nExample {i}: Image ID '{image_id}' is used by {len(products_with_image)} products")
    
#     # Show key product information including image path
#     examples = products_with_image[['item_id', 'item_name', 'image_path']].head(3)
#     print(examples)

#     # If there are more than 3 products, indicate that
#     if len(products_with_image) > 3:
#         print(f"... and {len(products_with_image) - 3} more products")
    
#     print("-" * 100)
